# 00 Overview - API Shortage Radar for BioChain Industries (start here)


**Outcome:** when a single-source API supplier fails an FDA inspection, planners see every exposed product, market and stockout date in minutes instead of 3-5 days, and every mitigation is proposed, approved and audited in one place. Estimated value **~$7.8M a year** (USE_CASE.md section 6). Synthetic data only; BioChain is fictional.

**Buyers:** VP Supply Chain (revenue at risk, stockout days, single-source exposure) and Supply Planning Lead (time to impact, days of cover, time to an approved decision).


## Demo disruption result (evidence 01-06)
- FDA Warning Letter for supplier V100103 exposes **29 SKUs** (25 single-source); first projected stockout **2027-01-11** (52 days); **21 SKUs** run out before an alternate source can be qualified; **$30.0M revenue at risk**.
- Late-PO model beats the baseline on a time-split holdout: **PR-AUC 0.562 vs 0.455** (EVAL_PROTOCOL bar: +0.05).
- Same query, two identities: supply exec sees 3 regions and revenue; EU planner sees EU only and NULL revenue.
- Genie 11/12 benchmark answers correct; routing agent 15/15 to the right tools; ai_query explanations 25/25 grounded.
- A new supplier quality file starts the job by itself (file-arrival trigger).
- Lakebase workflow: RLS blocks cross-region writes; approval needs an entitled approver and a rationale; audit log is append-only.


## Stage -> code -> evidence
| Stage | Code | Evidence |
|---|---|---|
| Lakeflow (Auto Loader, expectations, quarantine, conformed supplier codes) | src/pipeline/*.sql, src/data/generate.py | 01_pipeline.ipynb |
| Unity Catalog (region row filter, commercial masks, entitlements, gold-only grants) | src/governance/*.sql, src/pipeline/gold.sql | 02_governance.ipynb |
| ML + GenAI (late-PO model, serving endpoint, ai_query explanations) | src/ai/*.py | 03_ml.ipynb |
| Lakebase (synced impact, mitigation workflow, RLS, audit) | src/serve/*.sql, sync_lakebase.py | 04_lakebase.ipynb |
| Genie Agents + routing agent (Agent Bricks CLI, OBO, propose_mitigation tool) | src/genie/*.json, src/agent/agent/ | 05_genie_agent.ipynb |
| Databricks App (FastAPI + React, on behalf of user) | src/app/backend/main.py | 06_app.ipynb |


## How the stages connect
Model scores (ml.po_risk_score) feed gold days of cover and revenue at risk; gold impact is synced into Lakebase; the app and agent write decisions back to Lakebase; Genie and the app read only the governed gold layer. A new supplier quality file triggers the job: pipeline -> score -> refresh -> explain -> Lakebase sync.


## Decisions that matter (DECISIONS.md has all 12)
- Classic row filters and masks instead of governed-tag ABAC (no governed tags left in the workspace; ABAC is the production path).
- UC policies do not reach Lakebase, so entitlements are enforced again with Postgres RLS and approval triggers.
- Supervisor Agent and Knowledge Assistant are legacy (Nov 2026), so routing is a code agent over two Genie Agents with request-user auth.
- The model only counts if it beats a simple baseline written down before training (EVAL_PROTOCOL.md).
- Found in our own evidence: a human owner role bypasses Postgres RLS; app traffic runs as a service principal role without BYPASSRLS (BUILD_LOG.md).


## Files
README.md, USE_CASE.md, BUILD_PLAN.md, DECISIONS.md, EVAL_PROTOCOL.md, BUILD_LOG.md, VALIDATION.md, deck/shortage_radar_deck.md. Bundle: databricks.yml + resources/*.yml (quoted in VALIDATION.md).
